<a href="https://colab.research.google.com/github/Luizadraeger/URBAN-DATA-ANALYTICS/blob/pipeline/00_TFG_pipeline_BUILDINGHEIGHT_R01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Célula 3: Importa livrarias
import ee
import geemap
import geopandas as gpd
import os
import time
import pandas as pd

In [ ]:
# Célula 4: Autenticação e inicialização dos serviços Google
# Esta célula configura o acesso ao Google Drive e ao Google Earth Engine (EE).

# PASSOS NECESSÁRIOS:
# 1. Monte o Google Drive para leitura/escrita de arquivos.
# 2. Autentique sua conta Google para uso do Earth Engine.
# 3. Inicialize o Earth Engine com um projeto válido do Google Cloud.

# IMPORTANTE:
# - Antes de executar, crie um projeto no Google Cloud Console.
# - Ative a API do Earth Engine nesse projeto.
# - Substitua 'thermal-luizadraeger' pelo ID do seu projeto.
# - Durante a execução, será solicitado um token de autenticação no Colab.

from google.colab import drive

# Monta o Google Drive no ambiente do Colab
# force_remount=True garante remontagem caso já esteja conectado
try:
    drive.mount('/content/drive', force_remount=True)
except:
    drive.mount('/content/drive')

# ─── Autenticação no Google Earth Engine ─────────────────────────────────────
# Abre um fluxo de login para autorizar o acesso ao EE
import ee
ee.Authenticate()

# Inicializa o Earth Engine com o projeto especificado
# Substitua pelo seu próprio ID de projeto no Google Cloud
ee.Initialize(project='thermal-luizadraeger')

In [ ]:
def buildfooprint(lat, lon, radius_m=500):
    try:
        point = ee.Geometry.Point([lon, lat])
        region = point.buffer(radius_m).bounds()

        # Dataset de Alturas Temporais
        temporal_col = ee.ImageCollection('GOOGLE/Research/open-buildings-temporal/v1') \
                         .filterBounds(region) \
                         .sort('system:time_start', False)
        latest_raster = temporal_col.first()

        # Polígonos de Footprints
        buildings_fc = ee.FeatureCollection('GOOGLE/Research/open-buildings/v3/polygons') \
                         .filterBounds(region)

        # Amostragem de alturas
        sampled_fc = latest_raster.select('building_height').reduceRegions(
            collection=buildings_fc,
            reducer=ee.Reducer.median(),
            scale=1.0
        )

        features = sampled_fc.getInfo().get('features', [])
        building_list = []

        for f in features:
            props = f.get('properties', {})
            geom = f.get('geometry')
            if not geom: continue

            h_val = props.get('median')
            height = float(h_val) if (h_val and h_val > 0) else 3.5

            # Normalização de coordenadas
            coords = geom['coordinates'][0] if geom['type'] == 'Polygon' else geom['coordinates'][0][0]

            building_list.append({
                "geometry": coords,
                "height": height
            })

        return building_list
    except Exception as e:
        raise Exception("Erro no GEE: {}".format(str(e)))